# Objects, not blobs: language-grounded regions and object-level ablation

Clustering token embeddings gave contiguous but meaningless regions — one blob covering 55 of 81
patches, because "visually homogeneous" is not "an object". AdaVFM's Fig. 5 shows the fix:
**assign patches by matching them against *words*, not against each other.**

We use the stronger version of that. Luo et al. (ICCV 2025) measured both localisation routes:

```
Teacher LLM Attn.   56.14 / 74.81 / 73.88     <- grounding recall
CLIP / RemoteCLIP   20.14 / 30.41  ...        <- 2-3x worse
```

**LLM text->vision attention localises a named object far better than CLIP similarity** — and
grounding is the one thing our own experiments confirmed attention is good at (it was *causal
importance* it failed at). So we ground object names through the LLM's own attention.

## The pipeline

```
1. object names  <- content nouns from the QUESTION and the ANSWER
2. ground each name via text->vision attention at that word's token rows
3. assign every patch to its best-matching name    -> object masks
4. SANITY GATE: do the masks look like objects, and do different words give
   different maps?  If not, stop before spending GPU.
5. ablate whole objects
```

## What this finally lets us test

Your intuition, stated literally for the first time:

> *the question is about object A; the answer involves objects B, C; gaze lands on A, so predict B, C*

* **Test 3** — is the object whose removal hurts most the one the gaze is on, or a different one?
* **Test 4** — do objects named only in the **answer** matter more than objects named in the
  **question**? That is the "the answer involves *other* objects" claim, and it needs no ablation
  label at all if it holds — the answer text becomes free supervision.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install -U "transformers>=4.49" accelerate huggingface_hub safetensors pillow num2words matplotlib scipy nltk
!rm -rf /content/text_vision_attention_map
!git clone -q https://github.com/shubhamOjha1000/text_vision_attention_map.git /content/text_vision_attention_map
%cd /content/text_vision_attention_map

In [ ]:
import importlib.util, os, sys, math, time, gc, re
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.stats import wilcoxon

sys.path.insert(0, os.getcwd())

def _load(mod, rel):
    spec = importlib.util.spec_from_file_location(mod, os.path.join(os.getcwd(), rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m

S = _load("probe_smolvlm", "tests/probe_smolvlm.py")
import rater_selection as RS
import visual_selection as VS

N100  = "/content/drive/MyDrive/wearvqa_n100.pt"
SINKF = "/content/drive/MyDrive/sink_mask_smolvlm2_n12.pt"
OUT   = "/content/drive/MyDrive/wearvqa_objects.pt"
MODEL_ID = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
MAX_OBJ, MIN_PATCHES = 8, 2

assert os.path.exists(N100), f"{N100} missing - run colab_n100_scaleup.ipynb"
data  = [d for d in torch.load(N100, weights_only=False) if "gp" in d]
sinks = torch.load(SINKF, weights_only=False)["sink_mask"].bool()
L_v = data[0]["drops"].numel(); G = int(round(math.sqrt(L_v))); N = len(data)

import nltk
NLTK = True
for pkg in ("punkt", "punkt_tab", "averaged_perceptron_tagger",
            "averaged_perceptron_tagger_eng"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass
try:
    nltk.pos_tag(nltk.word_tokenize("a red dog runs"))
except Exception as e:
    NLTK = False
    print("nltk POS unavailable, falling back to content words:", e)

model, processor, device = S._load_smolvlm(MODEL_ID)
tokenizer = processor.tokenizer
print(f"{N} examples | L_v={L_v} ({G}x{G}) | nltk={NLTK}")

## 2. Object names, and grounding them through attention

In [ ]:
GENERIC = {"image", "picture", "photo", "thing", "things", "side", "front", "back",
           "top", "bottom", "left", "right", "part", "color", "colour", "shape", "type",
           "kind", "number", "purpose", "way", "use", "something", "area", "place"}

def nouns_of(text):
    if NLTK:
        out = [w.lower() for w, t in nltk.pos_tag(nltk.word_tokenize(text))
               if t.startswith("NN") and len(w) > 2]
    else:
        out = [RS._clean_token(w).lower() for w in re.findall(r"[A-Za-z]+", text)
               if len(w) > 2 and not RS.is_stopword_token(w)]
    seen, res = set(), []
    for w in out:
        if w not in seen and w not in GENERIC:
            seen.add(w); res.append(w)
    return res

def token_span(word, pieces):
    """all token indices whose characters fall inside any occurrence of the word."""
    text = "".join(pieces).lower()
    hits, start = [], 0
    while True:
        j = text.find(word, start)
        if j < 0:
            break
        hits.append((j, j + len(word))); start = j + 1
    idx, pos = [], 0
    for i, p in enumerate(pieces):
        a, b = pos, pos + len(p)
        if any(b > s and a < e for s, e in hits):
            idx.append(i)
        pos = b
    return idx

def build_inputs(image, question, answer=""):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    full = processor(text=prompt + answer, images=[image], return_tensors="pt").to(device)
    only = processor(text=prompt, images=[image], return_tensors="pt")
    return full, int(only["input_ids"].shape[1])

@torch.no_grad()
def answer_logprob(inp, n_prompt, attention_mask=None):
    kw = dict(inp)
    if attention_mask is not None:
        kw["attention_mask"] = attention_mask
    logits = model(**kw).logits[0].float()
    lp = torch.log_softmax(logits[:-1], dim=-1)
    return float(lp.gather(-1, inp["input_ids"][0, 1:, None]).squeeze(-1)[n_prompt-1:].sum())

@torch.no_grad()
def ground(inp):
    """post-softmax attention, averaged over heads and layers -> [L_text_all, L_v]."""
    patched = S._patch_eager_globals(S._make_raw_capturing_eager(None))
    try:
        model(**inp)
    finally:
        S._unpatch_eager_globals(patched)
    L = int(inp["input_ids"].shape[1])
    acc, n = None, 0
    for m in model.modules():
        p = getattr(m, "_post_attn", None)
        if p is not None and p.shape[-1] == L and p.shape[-2] == L:
            a = p[0].float().mean(0)                 # [L, L] head-average
            acc = a if acc is None else acc + a; n += 1
        for at in ("_raw_attn_scores", "_post_attn"):
            if hasattr(m, at):
                delattr(m, at)
    return acc / max(n, 1)

print("ok")

## 3. Build object masks — and gate on whether they are objects

Two automatic checks before any GPU is spent on ablation:

* **discrimination** — mean pairwise correlation between different objects' grounding maps.
  If every noun produces the same map, grounding is useless and nothing downstream means anything.
* **contiguity** — connected components per object mask.

In [ ]:
objs, t0 = [], time.time()
for i, d in enumerate(data):
    img = S.load_image(d["img_path"])
    inp, n_prompt = build_inputs(img, d["question"], d["answer"])
    ids = inp["input_ids"][0].cpu()
    iid = S._find_image_token_id(model, processor)
    pad = tokenizer.pad_token_id
    im = ids == iid
    tm = (ids != iid) & (ids != (pad if pad is not None else -10**9))
    vpos = torch.nonzero(im).squeeze(-1); tpos = torch.nonzero(tm).squeeze(-1)

    A = ground(inp)                                   # [L, L]
    toks = tokenizer.convert_ids_to_tokens(ids[tpos].tolist())
    pieces = [RS._detok_piece(t) for t in toks]

    qn, an = nouns_of(d["question"]), nouns_of(d["answer"])
    names = qn + [w for w in an if w not in qn]
    names = names[:MAX_OBJ]

    maps, kept, src = [], [], []
    for w in names:
        rows = token_span(w, pieces)
        rows = [r for r in rows if int(tpos[r]) >= 0]
        if not rows:
            continue
        m = A[tpos[rows]][:, vpos].mean(0)             # [L_v]
        m = m / m.sum().clamp_min(1e-9)
        maps.append(m); kept.append(w)
        src.append("Q" if w in qn else "A")
    if len(maps) < 2:
        objs.append(None); continue

    M = torch.stack(maps, 0)                           # [n_obj, L_v]
    lab = M.argmax(0)                                  # patch -> object
    objs.append(dict(idx=i, names=kept, src=src, M=M, lab=lab,
                     gaze_obj=int(lab[d["gp"]])))
    del A, inp
    gc.collect(); torch.cuda.empty_cache()
    if (i + 1) % 25 == 0:
        print(f"  grounded {i+1}/{N}  ({(time.time()-t0)/60:.1f} min)")

good = [o for o in objs if o is not None]
print(f"\nusable: {len(good)}/{N}   objects/image mean "
      f"{np.mean([len(o['names']) for o in good]):.1f}")

# --- GATE 1: do different words give different maps? ---
cors = []
for o in good:
    M = F.normalize(o["M"] - o["M"].mean(1, keepdim=True), dim=1)
    C = (M @ M.T).numpy()
    cors += [C[a, b] for a in range(len(C)) for b in range(a+1, len(C))]
print(f"mean pairwise correlation between object maps: {np.mean(cors):.3f}  "
      f"(near 1.0 = grounding does not discriminate)")

# --- GATE 2: contiguity ---
def n_comp(m2):
    seen = np.zeros_like(m2, dtype=bool); n = 0
    for i in range(G):
        for j in range(G):
            if m2[i, j] and not seen[i, j]:
                n += 1; st=[(i,j)]; seen[i,j]=True
                while st:
                    a,b = st.pop()
                    for da,db in ((1,0),(-1,0),(0,1),(0,-1)):
                        p,q = a+da,b+db
                        if 0<=p<G and 0<=q<G and m2[p,q] and not seen[p,q]:
                            seen[p,q]=True; st.append((p,q))
    return n
comps = [n_comp((o["lab"].reshape(G,G)==r).numpy())
         for o in good[:30] for r in range(len(o["names"]))
         if int((o["lab"]==r).sum()) >= MIN_PATCHES]
sz = [int((o["lab"]==r).sum()) for o in good for r in range(len(o["names"]))]
print(f"components per object mask: {np.mean(comps):.2f}   "
      f"mask size mean {np.mean(sz):.1f} min {np.min(sz)} max {np.max(sz)}")
print("\nGATE: correlation should be well below ~0.9, and the masks below should look")
print("like the named things. If not, stop here - grounding is not producing objects.")

In [ ]:
from PIL import Image as _I
fig, ax = plt.subplots(3, 4, figsize=(15, 11))
for a, o in zip(ax.ravel(), good[:12]):
    d = data[o["idx"]]; img = S.load_image(d["img_path"]); W, H = img.size
    seg = _I.fromarray((o["lab"].reshape(G,G).numpy() * (255 // max(len(o["names"]),1))
                        ).astype("uint8")).resize((W, H), _I.NEAREST)
    a.imshow(img); a.imshow(np.array(seg), cmap="tab10", alpha=0.55)
    gp = d["gp"]
    a.scatter([(gp % G + .5)/G*W], [(gp // G + .5)/G*H], marker="x", s=130, c="lime", linewidths=3)
    a.set_title(" / ".join(f"{n}[{s}]" for n, s in zip(o["names"], o["src"]))[:70], fontsize=6)
    a.axis("off")
plt.tight_layout(); plt.show()
print("titles list the grounded nouns, [Q] from the question and [A] from the answer.")

## 4. Ablate whole objects (plus a size-matched random control)

In [ ]:
rng = np.random.default_rng(0)
ok_idx = (~sinks[:L_v]).nonzero().squeeze(-1).numpy()

if os.path.exists(OUT):
    res = torch.load(OUT, weights_only=False)
    print(f"resuming from {len(res)}")
else:
    res = []
done = {r["idx"] for r in res}

t0 = time.time()
for o in good:
    if o["idx"] in done:
        continue
    d = data[o["idx"]]
    inp, n_prompt = build_inputs(S.load_image(d["img_path"]), d["question"], d["answer"])
    ids = inp["input_ids"][0].cpu()
    vpos = torch.nonzero(ids == S._find_image_token_id(model, processor)).squeeze(-1)
    base = answer_logprob(inp, n_prompt)

    def ablate(sel):
        if len(sel) == 0:
            return float("nan")
        am = inp["attention_mask"].clone(); am[0, vpos[np.asarray(sel)]] = 0
        return base - answer_logprob(inp, n_prompt, am)

    n_obj = len(o["names"])
    od, rd, sz = torch.full((n_obj,), float("nan")), torch.full((n_obj,), float("nan")), torch.zeros(n_obj)
    for r in range(n_obj):
        sel = ((o["lab"] == r) & (~sinks[:L_v])).nonzero().squeeze(-1).numpy()
        sz[r] = len(sel)
        if len(sel) < MIN_PATCHES:
            continue
        od[r] = ablate(sel)
        rd[r] = ablate(rng.choice(ok_idx, size=min(len(sel), len(ok_idx)), replace=False))
    res.append(dict(idx=o["idx"], base=base, obj_drops=od, rand_drops=rd, sizes=sz,
                    names=o["names"], src=o["src"], gaze_obj=o["gaze_obj"], lab=o["lab"]))
    if len(res) % 20 == 0:
        torch.save(res, OUT); print(f"  {len(res)}/{len(good)}  ({(time.time()-t0)/60:.1f} min)")

torch.save(res, OUT)
print(f"done in {(time.time()-t0)/60:.1f} min -> {OUT}")

## 5. TEST 1 — redundancy, and TEST 2 — did grouping beat size?

In [ ]:
ratios = []
for r in res:
    d = data[r["idx"]]
    for j in range(len(r["names"])):
        if not math.isfinite(float(r["obj_drops"][j])):
            continue
        sel = ((r["lab"] == j) & (~sinks[:L_v])).nonzero().squeeze(-1)
        indiv = float(d["drops"][sel].sum())
        if abs(indiv) < 1e-3:
            continue
        ratios.append(float(r["obj_drops"][j]) / indiv)
a = np.array(ratios)
print(f"TEST 1  super-additivity ratio (object vs sum of its patches)")
print(f"        median {np.median(a):.2f}   mean {a.mean():.2f}   "
      f">1.5 {(a>1.5).mean():.0%}   <1 {(a<1).mean():.0%}   n={len(a)}")
print("        >> 1 = patches inside an object cover for each other (redundancy)\n")

O = np.concatenate([r["obj_drops"].numpy() for r in res])
R = np.concatenate([r["rand_drops"].numpy() for r in res])
m = np.isfinite(O) & np.isfinite(R)
O, R = O[m], R[m]
print(f"TEST 2  object {O.mean():.3f}   random same-size {R.mean():.3f}   "
      f"diff {O.mean()-R.mean():+.3f}")
print(f"        paired p {wilcoxon(O, R)[1]:.3g}   object > random {(O>R).mean():.0%}   n={len(O)}")

## 6. TEST 3 — gaze object vs needed object.  TEST 4 — does the ANSWER name what matters?

In [ ]:
same, rank, outside = 0, [], []
for r in res:
    od = r["obj_drops"].clone(); od[torch.isnan(od)] = -1e9
    n_ok = int((od > -1e8).sum())
    if n_ok < 2:
        continue
    top = int(od.argmax()); gr = r["gaze_obj"]
    same += int(top == gr)
    rank.append(torch.argsort(od, descending=True).tolist().index(gr) + 1)
    tot = float(od[od > -1e8].clamp_min(0).sum() + 1e-9)
    outside.append(1 - max(float(od[gr]), 0) / tot)
n = len(rank)
print(f"TEST 3  top-damage object IS the gaze object : {same}/{n} ({same/n:.0%})")
print(f"        gaze object's mean rank {np.mean(rank):.2f} of "
      f"{np.mean([len(r['names']) for r in res]):.1f} objects")
print(f"        share of damage outside the gaze object: {np.mean(outside):.0%}\n")

q_d, a_d = [], []
for r in res:
    for j, s in enumerate(r["src"]):
        v = float(r["obj_drops"][j])
        if math.isfinite(v):
            (q_d if s == "Q" else a_d).append(v)
print(f"TEST 4  objects named in the QUESTION : mean drop {np.mean(q_d):.3f}  (n={len(q_d)})")
print(f"        objects named only in the ANSWER: mean drop {np.mean(a_d):.3f}  (n={len(a_d)})")
if len(q_d) > 5 and len(a_d) > 5:
    from scipy.stats import mannwhitneyu
    print(f"        Mann-Whitney p {mannwhitneyu(a_d, q_d, alternative='greater')[1]:.3g}")
print("        answer-only > question -> 'the answer involves OTHER objects' holds,")
print("        and the answer text is free supervision needing no ablation label.")

## 7. Verdict

**The gate comes first.** If the pairwise map correlation is near 1, or the masks in section 3 do
not look like the nouns that label them, nothing below is interpretable — grounding failed and the
fallback is SigLIP patch matching (AdaVFM's Fig. 5 mechanism) or a real segmenter.

**Test 1** — ratio >> 1 finally confirms the redundancy story that the Ward-cluster version rejected
(median 0.80). If real objects also give ~0.8, redundancy is definitively not the explanation and the
patch-level chain stands.

**Test 2** — objects must beat size-matched random sets. Ward regions did not (47-51%, a coin flip).
Language-grounded objects are a much stronger test of the same claim.

**Test 3** — the FRM premise at object level: is the object the answer needed the one the eye was on?

**Test 4** is the one your intuition predicts most directly. If objects named only in the answer
carry more damage than objects named in the question, then "the answer involves *other* objects" is
measured rather than assumed — and the answer text becomes a free label for training FRM, with no
ablation pass required at all.